# Wadjet v2 — Hieroglyph Detector (YOLO26s)

**Architecture**: YOLO26s (NMS-free end-to-end) → single-class detection ("hieroglyph")

| Property | Value |
|----------|-------|
| Input | `[1, 3, 640, 640]` float32, /255 normalized |
| Output | `[1, 300, 6]` = (batch, max_dets, [x1,y1,x2,y2,conf,cls]) |
| Classes | 1 ("hieroglyph") |
| Training data | 10,311 images (5 sources, YOLO format) |
| Quality gates | mAP50 ≥ 0.85, P/R ≥ 0.80, ONNX uint8 < 20MB |

**CRITICAL RULES:**
- **NO horizontal flip** (`fliplr=0.0`) — hieroglyph orientation = meaning
- **NO vertical flip** (`flipud=0.0`)
- float32 precision only (no AMP/mixed precision)
- End-to-end ONNX export (NMS-free, output `[1, 300, 6]`)

In [ ]:
# Cell 1: Install dependencies
!pip install -U ultralytics --quiet
!pip install -q onnxscript onnxruntime

In [ ]:
# Cell 2: Imports + Environment
import os
import sys
import json
import time
import shutil
import zipfile
from pathlib import Path

import torch
import onnx
import onnxruntime as ort
import numpy as np
from ultralytics import YOLO, settings

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} ({torch.version.cuda})")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"ONNX Runtime: {ort.__version__}")

import ultralytics
print(f"Ultralytics:  {ultralytics.__version__}")

In [ ]:
# Cell 3: Configuration
# ===================== TRAINING CONFIG =====================
MODEL_NAME   = "yolo26s.pt"       # COCO pretrained
IMG_SIZE     = 640
BATCH_SIZE   = 16
EPOCHS       = 150
PATIENCE     = 30
WARMUP_EP    = 5
LR0          = 0.01
LRF          = 0.01

# Augmentation (NO flips for hieroglyphs)
FLIPLR       = 0.0   # NO horizontal flip
FLIPUD       = 0.0   # NO vertical flip
MOSAIC       = 1.0
MIXUP        = 0.1
CLOSE_MOSAIC = 15
DEGREES      = 15.0
TRANSLATE    = 0.1
SCALE        = 0.5
PERSPECTIVE  = 0.001
HSV_H        = 0.015
HSV_S        = 0.7
HSV_V        = 0.4

# Quality gates
MIN_MAP50      = 0.85
MIN_PRECISION  = 0.80
MIN_RECALL     = 0.80
MAX_ONNX_MB    = 20
EXPECTED_SHAPE = (1, 300, 6)

# Output paths
OUTPUT_DIR   = "/kaggle/working"
PROJECT_DIR  = os.path.join(OUTPUT_DIR, "detector_train")

print("Config loaded.")
print(f"  Model: {MODEL_NAME}, ImgSize: {IMG_SIZE}, Batch: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}, Patience: {PATIENCE}")
print(f"  fliplr={FLIPLR}, flipud={FLIPUD} (NO FLIPS)")
print(f"  Quality gates: mAP50>={MIN_MAP50}, P>={MIN_PRECISION}, R>={MIN_RECALL}")

In [ ]:
# Cell 4: Auto-discover dataset path (deep recursive Kaggle search)
# The uploaded dataset may be nested or zipped inside Kaggle input.

DATA_ROOT = None
YAML_PATH = None

# Deep listing of /kaggle/input for debugging
print("=== Kaggle /kaggle/input/ deep listing (max 4 levels) ===", flush=True)
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth > 4:
        continue
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root)}/", flush=True)
    subindent = "  " * (depth + 1)
    for fname in files[:10]:
        fpath = os.path.join(root, fname)
        sz = os.path.getsize(fpath)
        print(f"{subindent}{fname} ({sz} bytes)", flush=True)
    if len(files) > 10:
        print(f"{subindent}... and {len(files)-10} more files", flush=True)
print("=== End listing ===", flush=True)

# Strategy: walk ALL of /kaggle/input looking for images/ + labels/ or data.yaml
for root, dirs, files in os.walk("/kaggle/input"):
    if "images" in dirs and "labels" in dirs:
        # Check if images/train exists
        img_train = os.path.join(root, "images", "train")
        if os.path.isdir(img_train):
            DATA_ROOT = root
            if "data.yaml" in files:
                YAML_PATH = os.path.join(root, "data.yaml")
            break
    elif "data.yaml" in files and "images" in dirs:
        DATA_ROOT = root
        YAML_PATH = os.path.join(root, "data.yaml")
        break

# If still not found, look for zip files to extract
if DATA_ROOT is None:
    print("No direct dataset found. Searching for zip files...", flush=True)
    for root, dirs, files in os.walk("/kaggle/input"):
        for fname in files:
            if fname.endswith(".zip"):
                zip_path = os.path.join(root, fname)
                extract_dir = os.path.join(OUTPUT_DIR, "dataset")
                print(f"Found zip: {zip_path} ({os.path.getsize(zip_path)} bytes)", flush=True)
                print(f"Extracting to {extract_dir}...", flush=True)
                os.makedirs(extract_dir, exist_ok=True)
                with zipfile.ZipFile(zip_path, "r") as zf:
                    zf.extractall(extract_dir)
                print("Extraction complete. Searching...", flush=True)
                for r2, d2, f2 in os.walk(extract_dir):
                    if "images" in d2 and "labels" in d2:
                        img_train = os.path.join(r2, "images", "train")
                        if os.path.isdir(img_train):
                            DATA_ROOT = r2
                            if "data.yaml" in f2:
                                YAML_PATH = os.path.join(r2, "data.yaml")
                            break
                if DATA_ROOT:
                    break
        if DATA_ROOT:
            break

# If we found DATA_ROOT but no YAML, search for data.yaml anywhere
if DATA_ROOT is not None and YAML_PATH is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "data.yaml" in files:
            YAML_PATH = os.path.join(root, "data.yaml")
            break
    if YAML_PATH is None:
        # Check extracted dir too
        extracted = os.path.join(OUTPUT_DIR, "dataset")
        if os.path.isdir(extracted):
            for root, dirs, files in os.walk(extracted):
                if "data.yaml" in files:
                    YAML_PATH = os.path.join(root, "data.yaml")
                    break

# If STILL no YAML, create one
if DATA_ROOT is not None and YAML_PATH is None:
    print("No data.yaml found, creating one...", flush=True)
    YAML_PATH = os.path.join(OUTPUT_DIR, "data.yaml")
    import yaml
    yaml_content = {
        "path": DATA_ROOT,
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": 1,
        "names": ["hieroglyph"]
    }
    with open(YAML_PATH, "w") as f:
        yaml.dump(yaml_content, f)

if DATA_ROOT is None:
    raise FileNotFoundError("Could not find dataset with images/ dir under /kaggle/input/")

print(f"\nDATA_ROOT: {DATA_ROOT}", flush=True)
print(f"YAML_PATH: {YAML_PATH}", flush=True)
print(f"Contents:  {os.listdir(DATA_ROOT)}", flush=True)


In [ ]:
# Cell 5: Prepare dataset YAML with absolute paths
# YOLO needs absolute paths in the yaml, not relative ones.
import yaml

# Read original yaml
with open(YAML_PATH, "r") as f:
    ds_config = yaml.safe_load(f)

print("Original data.yaml:")
print(json.dumps(ds_config, indent=2))

# Rewrite with absolute paths
ds_config["path"] = DATA_ROOT
ds_config["train"] = "images/train"
ds_config["val"]   = "images/val"
ds_config["test"]  = "images/test"

# Write fixed yaml to working dir
FIXED_YAML = os.path.join(OUTPUT_DIR, "hieroglyph_det.yaml")
with open(FIXED_YAML, "w") as f:
    yaml.dump(ds_config, f, default_flow_style=False)

print(f"\nFixed YAML written to: {FIXED_YAML}")
with open(FIXED_YAML, "r") as f:
    print(f.read())

In [ ]:
# Cell 6: Verify dataset integrity
splits = ["train", "val", "test"]
for split in splits:
    img_dir = os.path.join(DATA_ROOT, "images", split)
    lbl_dir = os.path.join(DATA_ROOT, "labels", split)
    if not os.path.isdir(img_dir):
        raise FileNotFoundError(f"Missing: {img_dir}")
    if not os.path.isdir(lbl_dir):
        raise FileNotFoundError(f"Missing: {lbl_dir}")
    n_img = len(os.listdir(img_dir))
    n_lbl = len(os.listdir(lbl_dir))
    print(f"{split:5s}: {n_img:,} images, {n_lbl:,} labels", flush=True)
    if n_img == 0:
        raise ValueError(f"{split} has 0 images!")
    if n_img != n_lbl:
        print(f"  WARNING: image/label count mismatch ({n_img} vs {n_lbl})")

# Quick label sanity check on first 100 train labels
lbl_dir = os.path.join(DATA_ROOT, "labels", "train")
issues = 0
for i, f in enumerate(sorted(os.listdir(lbl_dir))[:100]):
    with open(os.path.join(lbl_dir, f)) as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) != 5 or parts[0] != "0":
                issues += 1
if issues:
    print(f"WARNING: {issues} label issues in first 100 files")
else:
    print("Label format OK (class=0, 5 columns, checked 100 files)")

print("\nDataset verified.", flush=True)

In [ ]:
# Cell 7: Configure ultralytics settings
# Disable analytics, set project dir, disable wandb
settings.update({
    "runs_dir": PROJECT_DIR,
    "sync": False,
    "wandb": False,
})

# Disable AMP (mixed precision) globally for clean ONNX export
os.environ["YOLO_VERBOSE"] = "True"

print("Ultralytics settings configured.")
print(f"  runs_dir: {PROJECT_DIR}")

In [ ]:
# Cell 8: Train YOLO26s
#
# KeepAlive: ultralytics prints progress per epoch, but we add extra
# heartbeat prints to prevent IOPub timeout on long epochs.

import threading

# KeepAlive thread — prints a dot every 30s to prevent IOPub timeout
stop_keepalive = threading.Event()

def keepalive_thread():
    while not stop_keepalive.is_set():
        stop_keepalive.wait(30)
        if not stop_keepalive.is_set():
            print(".", end="", flush=True)

ka = threading.Thread(target=keepalive_thread, daemon=True)
ka.start()

print("="*60, flush=True)
print("STARTING YOLO26s TRAINING", flush=True)
print("="*60, flush=True)

t0 = time.time()

# Load COCO-pretrained YOLO26s
model = YOLO(MODEL_NAME)

# Train
results = model.train(
    data=FIXED_YAML,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    optimizer="auto",        # MuSGD (YOLO26 default)
    lr0=LR0,
    lrf=LRF,
    warmup_epochs=WARMUP_EP,
    # Augmentation — NO FLIPS
    fliplr=FLIPLR,
    flipud=FLIPUD,
    mosaic=MOSAIC,
    mixup=MIXUP,
    close_mosaic=CLOSE_MOSAIC,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    perspective=PERSPECTIVE,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    # Training settings
    amp=False,               # NO mixed precision — clean ONNX
    workers=2,               # Kaggle has limited CPU
    project=PROJECT_DIR,
    name="yolo26s_hiero",
    exist_ok=True,
    verbose=True,
    save=True,
    save_period=25,          # Checkpoint every 25 epochs
    plots=True,
)

stop_keepalive.set()
elapsed = (time.time() - t0) / 60
print(f"\nTraining complete in {elapsed:.1f} min", flush=True)

In [ ]:
# Cell 9: Find best.pt and evaluate on validation set

# Find the best.pt from training output
train_dir = os.path.join(PROJECT_DIR, "yolo26s_hiero")
best_pt = os.path.join(train_dir, "weights", "best.pt")

if not os.path.isfile(best_pt):
    # Search for it
    for root, dirs, files in os.walk(PROJECT_DIR):
        if "best.pt" in files:
            best_pt = os.path.join(root, "best.pt")
            break

if not os.path.isfile(best_pt):
    raise FileNotFoundError(f"best.pt not found under {PROJECT_DIR}")

print(f"Best model: {best_pt}")
print(f"Size: {os.path.getsize(best_pt) / 1024**2:.1f} MB")

# Validate on val set
best_model = YOLO(best_pt)
val_metrics = best_model.val(data=FIXED_YAML, split="val", verbose=True)

map50 = val_metrics.box.map50
map50_95 = val_metrics.box.map
precision = val_metrics.box.mp
recall = val_metrics.box.mr

print(f"\n{'='*50}")
print(f"VALIDATION RESULTS")
print(f"{'='*50}")
print(f"  mAP50:      {map50:.4f}  (gate: >= {MIN_MAP50})")
print(f"  mAP50-95:   {map50_95:.4f}")
print(f"  Precision:  {precision:.4f}  (gate: >= {MIN_PRECISION})")
print(f"  Recall:     {recall:.4f}  (gate: >= {MIN_RECALL})")

# Gate check
gates_passed = True
if map50 < MIN_MAP50:
    print(f"  FAIL: mAP50 {map50:.4f} < {MIN_MAP50}")
    gates_passed = False
if precision < MIN_PRECISION:
    print(f"  FAIL: Precision {precision:.4f} < {MIN_PRECISION}")
    gates_passed = False
if recall < MIN_RECALL:
    print(f"  FAIL: Recall {recall:.4f} < {MIN_RECALL}")
    gates_passed = False

if gates_passed:
    print("\n  ALL QUALITY GATES PASSED")
else:
    print("\n  WARNING: Some quality gates failed. Review results.")
    print("  Training will continue to export — review metrics before deploying.")

print(flush=True)

In [ ]:
# Cell 10: Evaluate on test set

print("Evaluating on TEST set...", flush=True)
test_metrics = best_model.val(data=FIXED_YAML, split="test", verbose=True)

test_map50 = test_metrics.box.map50
test_map50_95 = test_metrics.box.map
test_precision = test_metrics.box.mp
test_recall = test_metrics.box.mr

print(f"\n{'='*50}")
print(f"TEST RESULTS")
print(f"{'='*50}")
print(f"  mAP50:      {test_map50:.4f}")
print(f"  mAP50-95:   {test_map50_95:.4f}")
print(f"  Precision:  {test_precision:.4f}")
print(f"  Recall:     {test_recall:.4f}")
print(flush=True)

In [ ]:
# Cell 11: Export ONNX (end-to-end, NMS-free)

print("Exporting to ONNX (end-to-end, NMS-free)...", flush=True)

# ultralytics export handles ONNX internally
onnx_path = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    simplify=True,
    opset=17,
    dynamic=True,      # dynamic batch
    half=False,        # keep fp32
)

print(f"ONNX exported: {onnx_path}")
print(f"Size: {os.path.getsize(onnx_path) / 1024**2:.1f} MB")

In [ ]:
# Cell 12: Validate ONNX fp32 model

print("Validating ONNX fp32 model...", flush=True)

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

# Print I/O info
for inp in sess.get_inputs():
    print(f"  Input:  name={inp.name}, shape={inp.shape}, type={inp.type}")
for out in sess.get_outputs():
    print(f"  Output: name={out.name}, shape={out.shape}, type={out.type}")

# Run dummy inference
input_name = sess.get_inputs()[0].name
dummy = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
outputs = sess.run(None, {input_name: dummy})

print(f"\n  Output shape: {outputs[0].shape}")
print(f"  Output dtype: {outputs[0].dtype}")

# Check output shape
# YOLO26 end-to-end may output different shapes depending on version.
# Expected: [1, 300, 6] for end2end, or [1, N, 5+nc] for standard.
out_shape = outputs[0].shape
print(f"\n  Actual output shape: {out_shape}")

if len(out_shape) == 3 and out_shape[0] == 1:
    n_dets = out_shape[1]
    n_cols = out_shape[2]
    print(f"  Max detections: {n_dets}, Columns: {n_cols}")
    if n_cols >= 5:
        print(f"  Format: [batch, max_dets, (x1,y1,x2,y2,conf,...)]")
    print("  ONNX fp32 validation PASSED")
else:
    print(f"  WARNING: Unexpected output shape {out_shape}")
    print("  Will attempt to continue — check shape compatibility with deployment.")

print(flush=True)

In [ ]:
# Cell 13: Quantize to uint8

from onnxruntime.quantization import quantize_dynamic, QuantType

onnx_fp32 = onnx_path
onnx_uint8 = os.path.join(OUTPUT_DIR, "glyph_detector_uint8.onnx")

print("Quantizing to uint8...", flush=True)
quantize_dynamic(
    onnx_fp32,
    onnx_uint8,
    weight_type=QuantType.QUInt8,
)

fp32_size = os.path.getsize(onnx_fp32) / 1024**2
uint8_size = os.path.getsize(onnx_uint8) / 1024**2
print(f"  fp32:  {fp32_size:.1f} MB")
print(f"  uint8: {uint8_size:.1f} MB")
print(f"  Compression: {(1 - uint8_size/fp32_size)*100:.0f}%")

# Validate uint8 model
sess_q = ort.InferenceSession(onnx_uint8, providers=["CPUExecutionProvider"])
input_name_q = sess_q.get_inputs()[0].name
outputs_q = sess_q.run(None, {input_name_q: dummy})
print(f"  uint8 output shape: {outputs_q[0].shape}")

# Size gate
if uint8_size > MAX_ONNX_MB:
    print(f"  WARNING: uint8 model {uint8_size:.1f} MB > {MAX_ONNX_MB} MB budget")
else:
    print(f"  Size gate PASSED: {uint8_size:.1f} MB <= {MAX_ONNX_MB} MB")

print(flush=True)

In [ ]:
# Cell 14: Save model metadata

# Determine the actual output shape from uint8 model
actual_shape = list(outputs_q[0].shape)

metadata = {
    "model_name": "wadjet_hieroglyph_detector",
    "architecture": "YOLO26s (end-to-end, NMS-free)",
    "framework": "ultralytics",
    "task": "object_detection",
    "classes": ["hieroglyph"],
    "num_classes": 1,
    "input_size": IMG_SIZE,
    "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
    "input_format": "NCHW",
    "normalization": "divide_by_255",
    "input_range": [0.0, 1.0],
    "output_shape": actual_shape,
    "output_format": "[batch, max_detections, (x1, y1, x2, y2, confidence, class_id)]",
    "nms_free": True,
    "quantized": True,
    "quantization": "uint8_dynamic",
    "fp32_size_mb": round(fp32_size, 1),
    "uint8_size_mb": round(uint8_size, 1),
    "training": {
        "dataset": "nadermohamedcr7/wadjet-hieroglyph-detection",
        "train_images": len(os.listdir(os.path.join(DATA_ROOT, "images", "train"))),
        "val_images": len(os.listdir(os.path.join(DATA_ROOT, "images", "val"))),
        "test_images": len(os.listdir(os.path.join(DATA_ROOT, "images", "test"))),
        "epochs_configured": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "fliplr": FLIPLR,
        "flipud": FLIPUD,
        "amp": False,
        "pretrained": MODEL_NAME,
    },
    "metrics": {
        "val_mAP50": round(float(map50), 4),
        "val_mAP50_95": round(float(map50_95), 4),
        "val_precision": round(float(precision), 4),
        "val_recall": round(float(recall), 4),
        "test_mAP50": round(float(test_map50), 4),
        "test_mAP50_95": round(float(test_map50_95), 4),
        "test_precision": round(float(test_precision), 4),
        "test_recall": round(float(test_recall), 4),
    },
}

meta_path = os.path.join(OUTPUT_DIR, "model_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved: {meta_path}")
print(json.dumps(metadata, indent=2))

In [ ]:
# Cell 15: Copy artifacts to /kaggle/working for download

# List of output files
output_files = [
    (onnx_uint8, "glyph_detector_uint8.onnx"),
    (onnx_fp32, os.path.basename(onnx_fp32)),
    (best_pt, "best.pt"),
    (meta_path, "model_metadata.json"),
]

# Copy training plots if they exist
plots_dir = os.path.join(train_dir, "")
for fname in ["results.csv", "results.png", "confusion_matrix.png",
              "confusion_matrix_normalized.png", "P_curve.png",
              "R_curve.png", "PR_curve.png", "F1_curve.png"]:
    src = os.path.join(train_dir, fname)
    if os.path.isfile(src):
        dst = os.path.join(OUTPUT_DIR, fname)
        shutil.copy2(src, dst)
        output_files.append((dst, fname))

# Ensure all main artifacts are in /kaggle/working
for src, name in output_files[:4]:  # main 4 files
    dst = os.path.join(OUTPUT_DIR, name)
    if src != dst and os.path.isfile(src):
        shutil.copy2(src, dst)

print("\n" + "="*60)
print("OUTPUT FILES")
print("="*60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    # Skip deep training dirs
    depth = root.replace(OUTPUT_DIR, "").count(os.sep)
    if depth > 2:
        continue
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        rel = os.path.relpath(fpath, OUTPUT_DIR)
        if size > 1024*1024:
            print(f"  {rel:50s} {size/1024**2:8.1f} MB")
        elif size > 1024:
            print(f"  {rel:50s} {size/1024:8.1f} KB")
        else:
            print(f"  {rel:50s} {size:8d} B")

print(f"\nDownload from Kaggle output tab or run:")
print(f"  kaggle kernels output nadermohamedcr7/wadjet-hieroglyph-detector -p ./detector_output")
print(flush=True)

In [ ]:
# Cell 16: Final summary

print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"  Model:          YOLO26s (NMS-free end-to-end)")
print(f"  Dataset:        {metadata['training']['train_images']:,} train / {metadata['training']['val_images']:,} val / {metadata['training']['test_images']:,} test")
print(f"  ONNX uint8:     {uint8_size:.1f} MB")
print(f"  Output shape:   {actual_shape}")
print(f"")
print(f"  VAL  mAP50={map50:.4f}  P={precision:.4f}  R={recall:.4f}")
print(f"  TEST mAP50={test_map50:.4f}  P={test_precision:.4f}  R={test_recall:.4f}")
print(f"")

# Final gate summary
checks = [
    ("mAP50 >= 0.85",    map50 >= MIN_MAP50),
    ("Precision >= 0.80", precision >= MIN_PRECISION),
    ("Recall >= 0.80",    recall >= MIN_RECALL),
    (f"ONNX <= {MAX_ONNX_MB}MB", uint8_size <= MAX_ONNX_MB),
]

all_pass = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {name}")

print(f"")
if all_pass:
    print("  ALL GATES PASSED — Model ready for deployment!")
else:
    print("  SOME GATES FAILED — Review before deploying.")

print(f"\nDone.", flush=True)